# 07 — Integrator comparison: GSL vs Filon (vs GPU)

Runs the same BubbleMaster setup through multiple integrator backends and compares
the resulting GW spectra and wall-clock times.

**Workflow**
1. Run **Section 1** to write the shared setup file.
2. Run **Section 2** to generate SLURM scripts (one per backend).
3. Submit all scripts: `! for s in scripts/compare_*.sh; do sbatch $s; done`
4. After jobs complete, run **Sections 3–5** to compare spectra and timing.

To add the GPU backend later: uncomment the `gpu` entry in `BACKENDS` below.

In [ ]:
import sys
import re
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import _cutoff_times, BubbleMasterParams

config = pt.Config()
REPO_ROOT = Path(pt.__file__).parent.parent

# ---------------------------------------------------------------------------
# Validation point
# ---------------------------------------------------------------------------
LAMBDA_BAR = 0.84
D          = 55.55   # bubble centre-to-centre separation

params = BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                             how_often_ds=5, baby_steps=100, cutoff_type=0)

model  = pt.PhysicsModel(pt.Phi4Potential(LAMBDA_BAR), config)
t_coll = float(model.kinematics.collision_time(D))
gamma  = float(model.kinematics.Gamma(t_coll))
TAG    = f'lb{LAMBDA_BAR}_g{gamma:.4g}'

t_0, t_cut, t_m, t_max = _cutoff_times(D, params.cutoff_type, params.t_0_scal)
TIMES = np.linspace(t_m - 2*t_0, t_m + 2*t_0, 5)

RUN_DIR    = model.results_dir / 'integrator_compare' / TAG
SETUP_PATH = RUN_DIR / 'setup.h5'
SCRIPTS_DIR = REPO_ROOT / 'scripts'
LOGS_DIR    = REPO_ROOT / 'logs'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

print(f'TAG      = {TAG}')
print(f'd        = {D}   gamma = {gamma:.4f}')
print(f't_coll   = {t_coll:.3f}')
print(f't_cut    = {t_cut:.3f}   t_m = {t_m:.3f}   t_max = {t_max:.3f}')
print(f'run_dir  = {RUN_DIR}')

# ---------------------------------------------------------------------------
# Backends
# Add an entry here to include a new integrator in all subsequent cells.
# ---------------------------------------------------------------------------
BACKENDS = [
    dict(name='gsl',
         label='GSL adaptive (CPU)',
         binary=config.bubblemaster_bin,
         out_dir=RUN_DIR / 'out_gsl',
         color='#0072B2',
         nodes=1, cpus=128, time='02:00:00'),
    dict(name='filon',
         label=f'Filon N={2048} (CPU)',
         binary=config.bubblemaster_filon_bin,
         out_dir=RUN_DIR / 'out_filon',
         color='#D55E00',
         nodes=1, cpus=128, time='02:00:00'),
    # --- GPU backend (uncomment once bubblemaster_gpu is compiled) ---
    # dict(name='gpu',
    #      label='Filon (GPU)',
    #      binary=config.bubblemaster_gpu_bin,
    #      out_dir=RUN_DIR / 'out_gpu',
    #      color='#009E73',
    #      nodes=1, cpus=32, time='00:10:00'),
]

for b in BACKENDS:
    b['out_dir'].mkdir(parents=True, exist_ok=True)
    print(f"  [{b['name']}]  binary={b['binary']}")

## 1. Write shared setup file

In [ ]:
pt.write_2d_setup(model, gamma=gamma, times=TIMES,
                  path=SETUP_PATH, params=params)
print(f'Written: {SETUP_PATH}')

## 2. Generate SLURM scripts

In [ ]:
for b in BACKENDS:
    script_path = SCRIPTS_DIR / f'compare_{TAG}_{b["name"]}.sh'
    constraint = 'gpu' if b['name'] == 'gpu' else 'cpu'
    gpu_line   = '#SBATCH --gpus=1\n' if b['name'] == 'gpu' else ''

    slurm = f"""#!/bin/bash
#SBATCH --constraint={constraint}
#SBATCH --nodes={b['nodes']}
#SBATCH --tasks-per-node=1
#SBATCH --cpus-per-task={b['cpus']}
{gpu_line}#SBATCH --qos=regular
#SBATCH --time={b['time']}
#SBATCH --job-name=bm_{TAG}_{b['name']}
#SBATCH --output={LOGS_DIR}/bm_{TAG}_{b['name']}_%j.out

export OMP_NUM_THREADS={b['cpus']}
export OMP_PLACES=threads
export OMP_PROC_BIND=spread

mkdir -p {b['out_dir']}
/usr/bin/time --verbose srun {b['binary']} {SETUP_PATH} {b['out_dir']}/ --save-fields
"""
    script_path.write_text(slurm)
    print(f"[{b['name']}]  {script_path}")

print()
print('Submit all:')
for b in BACKENDS:
    print(f"  ! sbatch scripts/compare_{TAG}_{b['name']}.sh")

## 3. Load results

Reads `result_*.h5` files from each backend's output directory and interpolates
to `t_m`.  Cells that have not finished will print a warning and be skipped in
the comparison plots.

In [ ]:
def load_result(setup_path, out_dir):
    """Load spectrum interpolated to t_m.  Returns (w, spectrum, t_m)."""
    with h5py.File(setup_path, 'r') as f:
        t_m_setup = float(f.attrs['t_m'])
        times_all = f['times'][:]

    entries = []
    for p in Path(out_dir).glob('result_*.h5'):
        i_t = int(p.stem.split('_')[1])
        if i_t < len(times_all):
            entries.append((times_all[i_t], p))
    if not entries:
        raise FileNotFoundError(f'No result_*.h5 in {out_dir}')
    entries.sort()

    t_avail = np.array([e[0] for e in entries])
    if t_m_setup < t_avail[0] or t_m_setup > t_avail[-1]:
        raise RuntimeError(
            f't_m={t_m_setup:.4f} outside available range '
            f'[{t_avail[0]:.4f}, {t_avail[-1]:.4f}] '
            f'({len(entries)}/{len(times_all)} steps done)'
        )

    i = max(0, int(np.searchsorted(t_avail, t_m_setup)) - 1)
    i = min(i, len(entries) - 2)
    alpha = (t_m_setup - t_avail[i]) / (t_avail[i+1] - t_avail[i])

    with h5py.File(entries[i][1], 'r') as f:
        w      = f['w'][:]
        spec_a = f['spectrum'][:]
    with h5py.File(entries[i+1][1], 'r') as f:
        spec_b = f['spectrum'][:]

    return w, spec_a + alpha * (spec_b - spec_a), t_m_setup


results = {}
for b in BACKENDS:
    try:
        w, spec, t_m_val = load_result(SETUP_PATH, b['out_dir'])
        results[b['name']] = dict(w=w, spec=spec, t_m=t_m_val, **b)
        print(f"[{b['name']}]  loaded {len(spec)} frequency bins  t_m={t_m_val:.4f}")
    except (FileNotFoundError, RuntimeError) as e:
        print(f"[{b['name']}]  NOT READY: {e}")

## 4. Wall-clock timing

Reads `Maximum wall clock time` from the `/usr/bin/time --verbose` output in the
SLURM log files.

In [ ]:
def parse_walltime(log_dir, job_tag):
    """Return wall-clock seconds from the most recent matching SLURM log."""
    logs = sorted(Path(log_dir).glob(f'*{job_tag}*.out'))
    if not logs:
        return None
    text = logs[-1].read_text(errors='replace')
    # /usr/bin/time --verbose outputs "Elapsed (wall clock) time (h:mm:ss or m:ss): ..."
    m = re.search(r'Elapsed \(wall clock\) time.*?:\s+(\S+)', text)
    if not m:
        return None
    parts = m.group(1).split(':')
    parts = [float(x) for x in parts]
    if len(parts) == 2:
        return parts[0]*60 + parts[1]
    return parts[0]*3600 + parts[1]*60 + parts[2]


timings = {}
print(f'{'Backend':<12}  {'Wall time':>12}')
print('-' * 28)
for b in BACKENDS:
    job_tag = f"{TAG}_{b['name']}"
    secs    = parse_walltime(LOGS_DIR, job_tag)
    timings[b['name']] = secs
    if secs is not None:
        print(f"{b['name']:<12}  {secs/60:>10.1f} min")
    else:
        print(f"{b['name']:<12}  {'(log not found)':>12}")

## 5. Spectrum comparison

In [ ]:
if not results:
    print('No results loaded — run Section 3 after jobs complete.')
else:
    fig, (ax_spec, ax_diff) = plt.subplots(
        2, 1, figsize=(7, 7),
        gridspec_kw=dict(height_ratios=[2, 1], hspace=0.05),
        sharex=True,
    )

    # Use the first available backend as the reference for the relative difference
    ref_name = next(iter(results))
    ref      = results[ref_name]

    for name, r in results.items():
        timing_str = ''
        if timings.get(name) is not None:
            timing_str = f"  [{timings[name]/60:.1f} min]"
        label = r['label'] + timing_str

        kR = r['w'] * D
        ls = '-' if name == ref_name else '--'
        ax_spec.plot(kR, r['spec'], color=r['color'], lw=1.8,
                     linestyle=ls, label=label)

        if name != ref_name:
            # Interpolate onto reference w-grid for the residual
            spec_interp = np.interp(ref['w'], r['w'], r['spec'])
            rel_diff    = (spec_interp - ref['spec']) / (np.abs(ref['spec']) + 1e-30)
            ax_diff.plot(ref['w'] * D, rel_diff * 100,
                         color=r['color'], lw=1.5,
                         label=f"{r['label']} vs {ref['label']}")

    ax_spec.set_ylabel(r'$\Omega_\mathrm{GW}$ spectrum  [a.u.]')
    ax_spec.set_title(
        rf'$\bar\lambda={LAMBDA_BAR}$   $d={D}$   $\gamma={gamma:.3f}$   $t_m={ref["t_m"]:.3f}$',
        fontsize=9
    )
    ax_spec.legend(frameon=False, fontsize=8)
    ax_spec.set_yscale('log')
    ax_spec.set_xscale('log')

    ax_diff.axhline(0, color='grey', lw=0.8, ls=':')
    ax_diff.set_xlabel(r'$k R_*$')
    ax_diff.set_ylabel(r'relative diff  [%]')
    ax_diff.legend(frameon=False, fontsize=7)

    plt.tight_layout()
    plt.savefig(REPO_ROOT / f'compare_integrators_{TAG}.pdf', bbox_inches='tight')
    plt.show()

## 6. Per-frequency-bin residual table

In [ ]:
if len(results) >= 2:
    ref_name = next(iter(results))
    ref      = results[ref_name]

    header = f"{'w':>10}  {'kR':>10}  {ref_name:>14}"
    others = [n for n in results if n != ref_name]
    for n in others:
        header += f"  {n:>14}  {'rel diff [%]':>14}"
    print(header)
    print('-' * len(header))

    for j, (w_j, s_ref) in enumerate(zip(ref['w'], ref['spec'])):
        row = f"{w_j:10.4f}  {w_j*D:10.4f}  {s_ref:14.6e}"
        for n in others:
            r = results[n]
            s_n = np.interp(w_j, r['w'], r['spec'])
            rd  = (s_n - s_ref) / (abs(s_ref) + 1e-30) * 100
            row += f"  {s_n:14.6e}  {rd:+14.4f}"
        print(row)
else:
    print('Need at least two backends loaded to compute residuals.')